Experimentación y análisis exploratorio (carga inicial de datos y revisión visual).

In [1]:
import pandas as pd
from pathlib import Path

# Ruta robusta al proyecto: funciona sin importar si el cwd del kernel es la
# carpeta 'notebooks' o la raíz del proyecto.
NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = ROOT_DIR / 'data'

# Using the uploaded Excel file for epidemiological data
epidemiological_df = pd.read_excel(DATA_DIR / '01_raw' / 'casos_santander_bucaramanga_completo.xlsx')

# Using the uploaded CSV file for precipitation data
precipitation_df = pd.read_csv(DATA_DIR / '01_raw' / 'Precipitacion_Bucaramanga.csv')

# Using the uploaded CSV file for temperature data (was previously precipitation_df, corrected)
temperature_df = pd.read_csv(DATA_DIR / '01_raw' / 'Temperatura_Ambiente_del_Aire_20260628.csv')

# casos_santander_bucaramanga_completo.xlsx no trae ENFERMEDAD_CONSOLIDADA ni
# ARCHIVO_ORIGEN (esas dos columnas solo las agrega la celda "01. Extracción
# de ZIP" de 02_limpieza_transformacion.ipynb al construir
# consolidado_enfermedades_bucaramanga.xlsx). Si ese archivo ya existe en la
# raíz del proyecto, se recuperan esas dos columnas por clave compuesta (no
# hay una llave única simple: CONSECUTIVE solo tiene nulos/duplicados) para
# que epidemiological_df quede con las mismas 79 columnas que produce el
# flujo original de Colab.
archivo_consolidado = ROOT_DIR / 'consolidado_enfermedades_bucaramanga.xlsx'
if archivo_consolidado.exists():
    casos_detallados = pd.read_excel(archivo_consolidado, sheet_name='Casos_Detallados')
    columnas_extra = ['ENFERMEDAD_CONSOLIDADA', 'ARCHIVO_ORIGEN']
    if all(c in casos_detallados.columns for c in columnas_extra):
        llave = ['CONSECUTIVE', 'COD_EVE', 'FEC_NOT', 'SEMANA', 'ANO', 'COD_PRE', 'COD_SUB', 'EDAD', 'INI_SIN', 'NOM_UPGD']
        if all(c in epidemiological_df.columns for c in llave) and all(c in casos_detallados.columns for c in llave):
            for c in llave:
                epidemiological_df[c] = epidemiological_df[c].astype(str)
                casos_detallados[c] = casos_detallados[c].astype(str)
            filas_antes = len(epidemiological_df)
            epidemiological_df = epidemiological_df.merge(
                casos_detallados[llave + columnas_extra], on=llave, how='left'
            )
            if len(epidemiological_df) == filas_antes:
                print(f"✅ ENFERMEDAD_CONSOLIDADA y ARCHIVO_ORIGEN recuperados desde {archivo_consolidado.name}.")
            else:
                print(f"⚠️ El merge con {archivo_consolidado.name} generó un número de filas distinto ({filas_antes} -> {len(epidemiological_df)}); revisa la llave de cruce.")
        else:
            print(f"⚠️ Faltan columnas de la llave de cruce; se omite anexar ENFERMEDAD_CONSOLIDADA/ARCHIVO_ORIGEN.")
else:
    print(f"⚠️ No se encontró {archivo_consolidado.name} en la raíz del proyecto; epidemiological_df quedará sin ENFERMEDAD_CONSOLIDADA/ARCHIVO_ORIGEN.")

print("Epidemiological Data Head:")
display(epidemiological_df.head())

print("\nPrecipitation Data Head:")
display(precipitation_df.head())

print("\nTemperature Data Head:")
display(temperature_df.head())

# Persistir los DataFrames en disco para poder reutilizarlos en
# 02_limpieza_transformacion.ipynb. Cada notebook corre en su propio kernel,
# así que las variables no se comparten en memoria entre notebooks (a
# diferencia de Colab, donde todo corría en el mismo runtime).
# epidemiological_df se guarda en Excel (para poder revisarlo visualmente);
# precipitation_df y temperature_df se guardan en CSV, mucho más rápido para
# datasets grandes de cientos de miles de filas.
INTERIM_DIR = DATA_DIR / '02_intermediate'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

epidemiological_df.to_excel(INTERIM_DIR / 'epidemiological_df.xlsx', index=False)
precipitation_df.to_csv(INTERIM_DIR / 'precipitation_df.csv', index=False)
temperature_df.to_csv(INTERIM_DIR / 'temperature_df.csv', index=False)

print(f"\n✅ DataFrames guardados en {INTERIM_DIR.resolve()}")

✅ ENFERMEDAD_CONSOLIDADA y ARCHIVO_ORIGEN recuperados desde consolidado_enfermedades_bucaramanga.xlsx.
Epidemiological Data Head:


,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,NACIONALIDAD,...,DEPARTAMENTO_RESIDENCIA,MUNICIPIO_RESIDENCIA,DEPARTAMENTO_NOTIFICACION,MUNICIPIO_NOTIFICACION,PARTICIÓN,PARTICION,CONSECUTIVE_12,COD_EVE.1,ENFERMEDAD_CONSOLIDADA,ARCHIVO_ORIGEN
0,7232769.0,210,2020-02-27,9,2020,6800104268,1,15,1,170,...,SANTANDER,FLORIDABLANCA,SANTANDER,BUCARAMANGA,NaN,NaN,NaN,NaN,DENGUE,datos_al_ecosistema/Dengue_2010_2024/Datos_202...
1,7232789.0,210,2020-06-23,22,2020,6800104457,1,12,1,170,...,SANTANDER,GIRON,SANTANDER,BUCARAMANGA,NaN,NaN,NaN,NaN,DENGUE,datos_al_ecosistema/Dengue_2010_2024/Datos_202...
2,7233578.0,210,2020-03-08,11,2020,6800100701,2,11,1,170,...,SANTANDER,BUCARAMANGA,SANTANDER,BUCARAMANGA,NaN,NaN,NaN,NaN,DENGUE,datos_al_ecosistema/Dengue_2010_2024/Datos_202...
3,7233589.0,210,2020-03-04,10,2020,6827601366,13,39,1,170,...,SANTANDER,BUCARAMANGA,SANTANDER,FLORIDABLANCA,NaN,NaN,NaN,NaN,DENGUE,datos_al_ecosistema/Dengue_2010_2024/Datos_202...
4,7233675.0,210,2020-02-14,7,2020,6800100701,3,50,1,170,...,SANTANDER,BUCARAMANGA,SANTANDER,BUCARAMANGA,NaN,NaN,NaN,NaN,DENGUE,datos_al_ecosistema/Dengue_2010_2024/Datos_202...



Precipitation Data Head:


,CodigoEstacion,CodigoSensor,FechaObservacion,ValorObservado,NombreEstacion,Departamento,Municipio,ZonaHidrografica,Latitud,Longitud,DescripcionSensor,UnidadMedida
0,23195230,240,2017 Oct 29 04:40:00 PM,0,NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Precipitacion,mm
1,23195230,240,2018 May 24 10:10:00 AM,0,NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Precipitacion,mm
2,23195230,240,2018 Sep 21 11:00:00 AM,0,NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Precipitacion,mm
3,23195230,240,2015 Jun 24 09:10:00 AM,0,NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Precipitacion,mm
4,23195230,240,2019 May 27 11:20:00 PM,0,NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Precipitacion,mm



Temperature Data Head:


,CodigoEstacion,CodigoSensor,FechaObservacion,ValorObservado,NombreEstacion,Departamento,Municipio,ZonaHidrografica,Latitud,Longitud,DescripcionSensor,UnidadMedida
0,23195230,68,2020 Jan 21 02:00:00 PM,"28,6",NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Temp Aire 2 m,°C
1,23195230,68,2020 Jan 21 08:00:00 PM,"22,7",NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Temp Aire 2 m,°C
2,23195230,68,2020 Jan 21 09:00:00 PM,"22,7",NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Temp Aire 2 m,°C
3,23195230,68,2020 Jan 21 10:00:00 AM,"24,2",NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Temp Aire 2 m,°C
4,23195230,68,2020 Jan 21 04:00:00 AM,"19,8",NEOMUNDO - AUT,SANTANDER,BUCARAMANGA,MEDIO MAGDALENA,"7,102611111","-73,10730556",Temp Aire 2 m,°C



✅ DataFrames guardados en /home/agustine/Downloads/IA_PREDICTIVA/data/02_intermediate
